# 03 • Métriques, calibration et seuils

`[MÉTA | Formation 4-024 | Niveau Application | TP 03 | Mode CPU local]`

**Objectif :** Choisir une mesure utile et un seuil sur validation seulement.

**Temps indicatif :** 30 min et réutilisation J3. Ces temps sont répartis dans le conducteur, pas additionnés hors des 18 heures.

**Prérequis :** TP 02.

**Preuves de réussite :** Matrice interprétée, AP distinguée de PR-AUC, capacité chiffrée.

**Sources :** R04.

Les jeux métier sont synthétiques. Aucun fichier personnel ou fiscal réel ne doit être chargé. Les résultats obtenus ici ne constituent pas une validation métier.

**Mode d’emploi :** exécuter les cellules dans l’ordre. Les cellules d’exercice du cahier apprenant sont à compléter ; le corrigé contient le code et des résultats de référence sur CPU.

In [ ]:
from pathlib import Path
import sys, os, json
# Chercher le kit depuis le répertoire du notebook ou celui de lancement.
HERE = Path.cwd().resolve()
TP_ROOT = next((p for p in [HERE, *HERE.parents] if (p / "modules" / "atelier.py").exists()), None)
if TP_ROOT is None:
    raise FileNotFoundError("Ouvrir ce notebook depuis le dossier 03_Travaux_pratiques du kit décompressé.")
sys.path.insert(0, str(TP_ROOT / "modules"))
os.environ.setdefault("KERAS_BACKEND", "torch")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch
from torch import nn
from atelier import *
seed_all(42)
print("Moteur disponible :", torch.__version__, "| Données :", DATA)


## 1. Scores de référence
On reprend un modèle linéaire rapide sur les mêmes données. Les principes d’évaluation ne dépendent pas du choix neuronal.

In [ ]:
d=split_tabular();m=LogisticRegression(max_iter=300).fit(*d['train'])
X,y=d['validation'];score=m.predict_proba(X)[:,1]
from sklearn.metrics import ConfusionMatrixDisplay,precision_recall_curve,roc_curve,auc
print(pd.DataFrame([binary_metrics(y,score,t) for t in [.2,.5,.8]]).round(3))
ConfusionMatrixDisplay.from_predictions(y,score>=.5,display_labels=['non','oui'])
plt.title('Validation : seuil 0,5');plt.tight_layout();plt.savefig(RESULTS/'03_confusion.png',dpi=150)

## 2. Courbe précision-rappel et AP
L’average precision (AP, précision moyenne) n’est pas strictement l’aire trapézoïdale sous la courbe précision-rappel. Afficher les deux noms exactement pour éviter une confusion dans la restitution. La ROC n’est pas supprimée ; la courbe PR répond à une autre question utile aux classes rares.

In [ ]:
# EXERCICE À COMPLÉTER
# Utiliser precision_recall_curve et average_precision_score. Ne pas nommer AP « aire trapézoïdale ».
# La correction est fournie séparément au formateur.
raise NotImplementedError("Compléter cette cellule puis relancer avant de poursuivre.")

## 3. Choisir un seuil opérationnel
Contrainte fictive : au plus 50 alertes sur cette validation. Chercher parmi les seuils des scores disponibles celui maximisant le rappel dans cette capacité. En cas d’égalité, privilégier le seuil le plus élevé. Ce choix ne garantit pas 50 alertes dans un futur lot de taille ou de distribution différente.

In [ ]:
# EXERCICE À COMPLÉTER
# Filtrer les configurations par alertes<=50 avant de maximiser le rappel. Garder le test fermé.
# La correction est fournie séparément au formateur.
raise NotImplementedError("Compléter cette cellule puis relancer avant de poursuivre.")

## 4. Calibration
Comparer fréquence observée et score moyen par intervalle. Des intervalles peu peuplés sont instables. Le graphique décrit ce modèle et ce jeu, sans garantir des probabilités fiables en production.

In [ ]:
from sklearn.calibration import calibration_curve
observes,predits=calibration_curve(y,score,n_bins=6,strategy='quantile')
fig,ax=plt.subplots(figsize=(6,4));ax.plot(predits,observes,marker='o',label='Modèle');ax.plot([0,1],[0,1],linestyle=':',label='Référence idéale')
ax.set(xlabel='Score moyen',ylabel='Fréquence positive',title='Fiabilité sur validation');ax.legend();fig.tight_layout();fig.savefig(RESULTS/'03_calibration.png',dpi=150)
save_result('03_metr iques'.replace(' ',''),{'AP':ap,'PR_AUC_trapezoidale':pr_auc,'seuil_validation':choix,'test_ouvert':False})

## 5. Questions à restituer
Pourquoi l’accuracy peut-elle être trompeuse ? Qu’est-ce qu’une alerte utile ? Que change le seuil, et que ne change-t-il pas ? Pourquoi la calibration et le classement ne sont-ils pas synonymes ? **Extension :** ajouter un coût fictif explicite aux faux positifs et faux négatifs, puis comparer au critère de capacité.